<a href="https://colab.research.google.com/github/dkayikci/COINS/blob/main/GRAD_CAM_Metrics_for_basemodels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# GRAD-CAM CONCENTRATION METRICS (Heritage revision, reviewer comment 5)
# Loads the interpretability checkpoints saved by the main pipeline (model_<DATASET>_<BACKBONE>_
# <MODE>_<REGIME>.pt), recomputes Grad-CAM exactly as the main pipeline does (target layer
# features[-1], same scaler, same input), and measures how concentrated or spread the attention is:
#   entropy   : normalised spatial entropy of the Grad-CAM map (0 = one pixel, 1 = uniform)
#   top10     : share of the total activation held by the 10% most active pixels
#   periph    : share of the total activation outside a central disc (radius 0.6 x half-width)
#   periph50/70 : the same with radius 0.5 and 0.7 (robustness of the disc choice)
# No model is trained. Results are written to OUT_DIR.
# =====================================================================================

BASE    = "/content/drive/MyDrive/UABandThesis/ThesisPhD/Original_Codes_Chapter5/"
DATASETS = ["MetaCoinsChro", "MetaCoinsGeo"]
BACKBONES = ["resnet18", "efficientnet_b0"]
MODES   = ["balanced", "legacy"]          # Table 8 / Figures 4-6 use "balanced"
REGIMES = ["partial", "full"]             # partial = Stage 1, full = Stage 2
OUT_DIR = BASE + "gradcam_metrics"

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, time, warnings, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
import scipy, scipy.signal, scipy.stats, pywt
from scipy.signal import hilbert
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG = 224; FIXED_DIM = 200; PROPS = (0.40, 0.30, 0.30)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

# ---------------- signal descriptors (verbatim from the main pipeline) ----------------
NATIVE = {"wavelet": 112, "hht": 200, "laws": 175}
LAWS_1D = {"L5": np.array([1,4,6,4,1]), "E5": np.array([-1,-2,0,2,1]), "S5": np.array([-1,0,2,0,-1]),
           "W5": np.array([-1,2,0,-2,1]), "R5": np.array([1,-4,6,-4,1])}
LAWS_NAMES = [f"{a}x{b}" for a in LAWS_1D for b in LAWS_1D]
def _entropy(x):
    p = np.abs(np.asarray(x)).flatten(); t = p.sum()
    if t <= 0: return 0.0
    p = p/t; p = p[p > 0]
    return float(-np.sum(p*np.log2(p)))
def _stats7(v):
    v = np.asarray(v).flatten()
    return [float(np.mean(np.abs(v))), float(np.std(v)), float(np.var(v)), float(np.mean(v**2)), _entropy(v),
            float(scipy.stats.skew(v)), float(scipy.stats.kurtosis(v))]
def _emd(sig, n=5):
    imfs=[]; res=np.asarray(sig, dtype=float).copy()
    for _ in range(n):
        if len(res) < 10: break
        pk = scipy.signal.find_peaks(res)[0]; vl = scipy.signal.find_peaks(-res)[0]
        if len(pk) < 2 or len(vl) < 2: break
        up = np.interp(range(len(res)), pk, res[pk]); lo = np.interp(range(len(res)), vl, res[vl])
        imf = res - (up+lo)/2; imfs.append(imf); res = res - imf
        if np.std(imf) < 0.01: break
    return imfs
def sampled_line_indices(gray, axis, n_lines=5):
    n = gray.shape[0] if axis == "row" else gray.shape[1]
    return list(range(0, n, max(1, n//10)))[:n_lines]
def wavelet_block(gray):
    f = []
    for lvl in range(1, 5):
        c = pywt.wavedec2(gray, "db4", level=lvl)
        for band in (c[0], c[1][0], c[1][1], c[1][2]): f += _stats7(band)
    return np.array(f)
def hht_block(gray, n_imfs=5, n_lines=5):
    f = []
    for axis in ["row","col"]:
        idx = sampled_line_indices(gray, axis, n_lines)
        for li in range(n_lines):
            imfs = (_emd(gray[idx[li],:] if axis=="row" else gray[:,idx[li]], n_imfs) if li < len(idx) else [])
            for k in range(n_imfs):
                if k < len(imfs):
                    an = hilbert(imfs[k]); a = np.abs(an); fr = np.diff(np.unwrap(np.angle(an)))/(2*np.pi)
                    f += [float(np.mean(a)), float(np.std(a)), float(np.mean(fr)) if len(fr) else 0.0,
                          float(np.std(fr)) if len(fr) else 0.0]
                else: f += [0.0, 0.0, 0.0, 0.0]
    return np.array(f)
def laws_block(gray):
    f = []
    for name in LAWS_NAMES:
        a, b = name.split("x")
        f += _stats7(np.abs(scipy.signal.convolve2d(gray, np.outer(LAWS_1D[a], LAWS_1D[b]), mode="valid")))
    return np.array(f)
def legacy_blocks(gray):
    c = pywt.wavedec2(gray, "db4", level=4); w=[]
    for i, coeff in enumerate(c):
        if i == 0:
            w += [np.mean(coeff), np.std(coeff), np.var(coeff), scipy.stats.skew(coeff.flatten()), scipy.stats.kurtosis(coeff.flatten())]
        else:
            for d in coeff:
                w += [np.mean(d), np.std(d), np.var(d), np.max(d), np.min(d), scipy.stats.skew(d.flatten()), scipy.stats.kurtosis(d.flatten())]
    h=[]
    for axis in ["row","col"]:
        for i in sampled_line_indices(gray, axis):
            for imf in _emd(gray[i,:] if axis=="row" else gray[:,i]):
                an = hilbert(imf); a = np.abs(an); fr = np.diff(np.unwrap(np.angle(an)))/(2*np.pi)
                h += [np.mean(a), np.std(a), np.mean(fr) if len(fr) else 0, np.std(fr) if len(fr) else 0]
    l=[]
    for name in LAWS_NAMES:
        a, b = name.split("x")
        t = np.abs(scipy.signal.convolve2d(gray, np.outer(LAWS_1D[a], LAWS_1D[b]), mode="valid"))
        l += [np.mean(t), np.std(t), np.var(t), np.max(t), np.min(t), scipy.stats.skew(t.flatten()), scipy.stats.kurtosis(t.flatten())]
    return np.array(w), np.array(h), np.array(l)
def allocate(budget, props=PROPS, native=NATIVE):
    keys = ["wavelet","hht","laws"]
    want = {k:int(round(budget*p)) for k,p in zip(keys, props)}
    want[keys[-1]] = budget - want[keys[0]] - want[keys[1]]
    for _ in range(4):
        over = 0
        for k in keys:
            if want[k] > native[k]: over += want[k]-native[k]; want[k] = native[k]
        if over == 0: break
        room = {k: native[k]-want[k] for k in keys}; tr = sum(room.values())
        if tr == 0: break
        for k in keys: want[k] += min(room[k], int(round(over*room[k]/tr)))
    return want
def fit_len(v, n):
    if len(v) > n: return v[:n]
    if len(v) < n: return np.concatenate([v, np.zeros(n-len(v))])
    return v
def build_vector(blocks, budget, mode):
    w,h,l = blocks
    if mode == "legacy": return fit_len(np.concatenate([w,h,l]), budget)
    a = allocate(budget)
    return np.concatenate([fit_len(w,a["wavelet"]), fit_len(h,a["hht"]), fit_len(l,a["laws"])])

# ---------------- model (main pipeline's SignalNet) ----------------
def build_trunk(name, regime):
    if name.startswith("efficientnet"):
        trunk = models.efficientnet_b0(weights=None).features
    else:
        trunk = nn.Sequential(*list(models.resnet18(weights=None).children())[:-2])
    if regime == "partial":
        for p in trunk.parameters(): p.requires_grad = False
        for p in trunk[-1].parameters(): p.requires_grad = True
    return trunk, (1280 if name.startswith("efficientnet") else 512)

class SignalNet(nn.Module):
    def __init__(self, num_classes, sdim, backbone, regime):
        super().__init__()
        self.features, vdim = build_trunk(backbone, regime)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.signal = nn.Sequential(nn.Linear(sdim,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Sequential(nn.Linear(vdim+64,256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256,num_classes))
    def forward(self, x, s):
        v = self.pool(self.features(x)).flatten(1)
        return self.head(torch.cat([v, self.signal(s)], dim=1))

class GradCAM:   # verbatim logic of the main pipeline
    def __init__(self, model, target_layer):
        self.model=model; self.acts=None; self.grads=None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,"acts",o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,"grads",go[0].detach()))
    def __call__(self, x, s):
        self.model.zero_grad()
        x = x.clone().requires_grad_(True)
        out = self.model(x, s); cls = out.argmax(1)
        out.gather(1, cls.view(-1,1)).sum().backward()
        w = self.grads.mean(dim=(2,3), keepdim=True)
        cam = F.relu((w*self.acts).sum(1))
        cam = cam / (cam.amax(dim=(1,2), keepdim=True) + 1e-8)
        return cam.cpu().numpy(), cls.cpu().numpy()

# ---------------- concentration metrics ----------------
YY, XX = np.mgrid[0:IMG, 0:IMG]; RR = np.sqrt((YY-(IMG-1)/2)**2 + (XX-(IMG-1)/2)**2) / (IMG/2)
def metrics(cam224):
    m = np.clip(cam224, 0, None).astype(float); tot = m.sum()
    if tot <= 0: return dict(entropy=np.nan, top10=np.nan, periph=np.nan, periph50=np.nan, periph70=np.nan)
    p = (m/tot).ravel(); nz = p[p > 0]
    ent = float(-(nz*np.log(nz)).sum() / np.log(p.size))
    k = int(0.10*p.size); top10 = float(np.sort(p)[::-1][:k].sum())
    per = lambda r: float(m[RR > r].sum()/tot)
    return dict(entropy=ent, top10=top10, periph=per(0.6), periph50=per(0.5), periph70=per(0.7))

# ---------------- run ----------------
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1); STD = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
tfm = transforms.Compose([transforms.Resize((IMG,IMG)), transforms.ToTensor(),
                          transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
rows = []
for ds_name in DATASETS:
    root = BASE + ds_name + "/"
    splits = {s: ImageFolder(os.path.join(root, s), transform=tfm) for s in ["train","test"]}
    classes = splits["train"].classes
    images, labels, aligned, legacy, split_of, names = [], [], [], [], [], []
    t0 = time.time()
    for sname, ds in splits.items():
        for i in range(len(ds)):
            x, y = ds[i]; images.append(x); labels.append(y); split_of.append(sname)
            names.append(os.path.basename(ds.samples[i][0]))
            gray = np.mean(torch.clamp(x*STD+MEAN, 0, 1).permute(1,2,0).numpy(), axis=2)
            aligned.append((wavelet_block(gray), hht_block(gray), laws_block(gray))); legacy.append(legacy_blocks(gray))
    images = torch.stack(images); labels = np.array(labels); split_of = np.array(split_of)
    train_idx = np.where(split_of=="train")[0]; test_idx = np.where(split_of=="test")[0]
    print(f"\n{ds_name}: {len(labels)} images, descriptors {time.time()-t0:.0f}s")
    for bb, mode, regime in itertools.product(BACKBONES, MODES, REGIMES):
        ckpt = os.path.join(root, f"model_{ds_name}_{bb}_{mode}_{regime}.pt")
        if not os.path.exists(ckpt):
            print(f"  MISSING {os.path.basename(ckpt)} — skipped"); continue
        V = np.array([build_vector(b, FIXED_DIM, mode) for b in (legacy if mode=="legacy" else aligned)])
        V = StandardScaler().fit(V[train_idx]).transform(V)
        model = SignalNet(len(classes), FIXED_DIM, bb, regime).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device)); model.eval()
        cammer = GradCAM(model, model.features[-1])
        correct = 0; order = 0
        for i in test_idx:
            x = images[i:i+1].to(device); s = torch.tensor(V[i:i+1], dtype=torch.float32).to(device)
            heat, pred = cammer(x, s)
            ok = int(pred[0]) == int(labels[i]); correct += ok
            if not ok: continue
            order += 1
            cam224 = F.interpolate(torch.tensor(heat[None].astype(np.float32)), size=(IMG,IMG),
                                   mode="bilinear", align_corners=False)[0,0].numpy()
            rows.append(dict(task="chronological" if "Chro" in ds_name else "geographical", backbone=bb,
                             stage=1 if regime=="partial" else 2, layout=mode, image=names[i],
                             true_class=classes[labels[i]], rank_among_correct=order, **metrics(cam224)))
        print(f"  {bb:16s} {mode:9s} stage {1 if regime=='partial' else 2}: single-model test accuracy "
              f"{correct/len(test_idx):.4f} ({correct}/{len(test_idx)} correct)")
        del model; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "gradcam_metrics_per_image.csv"), index=False)

# ---------------- summaries ----------------
M = ["entropy", "top10", "periph"]
summ = df.groupby(["layout","backbone","stage","task"])[M].agg(["mean","std","count"]).round(3)
print("\nPER CONFIGURATION (mean, SD, n correctly classified test images)"); print(summ.to_string())
summ.to_csv(os.path.join(OUT_DIR, "gradcam_metrics_by_configuration.csv"))

print("\nCHRONOLOGICAL vs GEOGRAPHICAL (same backbone, stage, layout); Mann-Whitney U, two-sided")
tests = []
for (lay, bb, st), g in df.groupby(["layout","backbone","stage"]):
    c = g[g.task=="chronological"]; ge = g[g.task=="geographical"]
    if len(c) < 2 or len(ge) < 2: continue
    for m in M + ["periph50", "periph70"]:
        u, p = mannwhitneyu(c[m], ge[m], alternative="two-sided")
        tests.append(dict(layout=lay, backbone=bb, stage=st, metric=m, chrono_mean=round(c[m].mean(),3),
                          geo_mean=round(ge[m].mean(),3), diff_geo_minus_chrono=round(ge[m].mean()-c[m].mean(),3),
                          n_chrono=len(c), n_geo=len(ge), U=u, p=round(p,4)))
tests = pd.DataFrame(tests); print(tests.to_string(index=False))
tests.to_csv(os.path.join(OUT_DIR, "gradcam_metrics_chrono_vs_geo.csv"), index=False)
print(f"\nFiles written to {OUT_DIR}")

Mounted at /content/drive

MetaCoinsChro: 40 images, descriptors 53s
  resnet18         balanced  stage 1: single-model test accuracy 0.7500 (6/8 correct)
  resnet18         balanced  stage 2: single-model test accuracy 0.6250 (5/8 correct)
  resnet18         legacy    stage 1: single-model test accuracy 0.8750 (7/8 correct)
  resnet18         legacy    stage 2: single-model test accuracy 0.6250 (5/8 correct)
  efficientnet_b0  balanced  stage 1: single-model test accuracy 0.8750 (7/8 correct)
  efficientnet_b0  balanced  stage 2: single-model test accuracy 0.8750 (7/8 correct)
  efficientnet_b0  legacy    stage 1: single-model test accuracy 0.8750 (7/8 correct)
  efficientnet_b0  legacy    stage 2: single-model test accuracy 0.6250 (5/8 correct)

MetaCoinsGeo: 80 images, descriptors 103s
  resnet18         balanced  stage 1: single-model test accuracy 0.8125 (13/16 correct)
  resnet18         balanced  stage 2: single-model test accuracy 0.8750 (14/16 correct)
  resnet18         legac